In [1]:
# %% [markdown]
# # 0) セットアップ
# - Qwen2.5-3B-Instruct をロード
# - tokenizer の pad/eos を整備
# - 乱数固定（再現性）

# %%
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, random, os
from typing import List, Dict

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"   # お手元のローカルに置き換え可

# 再現性
random.seed(42)
torch.manual_seed(42)

# Tokenizer & Model
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

device = next(model.parameters()).device
print("Loaded:", MODEL_ID, "| device:", device, "| dtype:", dtype)

# 共通: 生成ユーティリティ
def generate_text(prompt, max_new_tokens=64, **gen_kwargs):
    input_ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
        **gen_kwargs
    )
    # プロンプト差分のみ返す
    gen = tok.decode(out[0][input_ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-14B-Instruct | device: cuda:0 | dtype: torch.bfloat16


In [2]:
# %% [markdown]
# # 1) ベースライン：普通に短い文を出す
# まずは通常の自由生成で短い日本語文を出して、動作確認。

# %%
prompt = "次の質問に1文で答えて。質問: 生成AI研究で最近気になるテーマは？"
print(generate_text(prompt, max_new_tokens=1000, do_sample=False, temperature=0.0))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 最近の生成AI研究では、倫理的な課題やバイアスの問題が大きな注目を集めています。これにより、生成AIの開発と利用における公平性と透明性を確保するための新たな手法やフレームワークの探索が重要なテーマとなっています。また、生成AIの持続可能性やエネルギー効率の向上も重要な研究テーマとして挙げられます。これらの中で特に注目すべきは、生成AIの倫理的課題とバイアスの問題への対応です。生成AIの倫理的課題とバイアスの問題への対応が最近の生成AI研究で最も注目されているテーマです。ただし、1文で簡潔に表現すると、以下のようにまとめられます：

生成AIの倫理的課題とバイアスの問題への対応が最近の主要な研究テーマです。 

この回答は、生成AI研究における最も重要な現在の課題である倫理的課題とバイアスの問題への取り組みを強調しています。これは、生成AI技術の進歩とともに浮上してきた重要な問題であり、その解決策を見つけることは、生成AIの将来にとって不可欠です。また、この問題に対する研究は、生成AIの信頼性と社会的受け入れを高めるために不可欠です。ただし、エネルギー効率や持続可能性など他の重要なテーマも存在しますが、倫理的課題とバイアスの問題は特に注目されています。 

ただし、厳密に1文で簡潔に表現すると：

生成AIの倫理的課題とバイアスの問題への対応が最近の主要な研究テーマです。 

これが最も適切な回答となります。この文は、生成AI研究における最も重要な現在の課題である倫理的課題とバイアスの問題への取り組みを強調しています。これは、生成AI技術の進歩とともに浮上してきた重要な問題であり、その解決策を見つけることは、生成AIの将来にとって不可欠です。ただし、エネルギー効率や持続可能性など他の重要なテーマも存在しますが、倫理的課題とバイアスの問題は特に注目されています。 

以上より、最も適切な1文での回答は以下の通りです：

生成AIの倫理的課題とバイアスの問題への対応が最近の主要な研究テーマです。 

この回答は、生成AI研究における最も重要な現在の課題である倫理的課題とバイアスの問題への取り組みを強調しています。ただし、厳格に1文で表現する場合、この回答が最も適切です。 

最終的に、最も簡潔で適切な回答は以下の通りです：

生成AIの倫理的課題とバイアスの問題への対応

In [3]:
# %% [markdown]
# # 2) 方式A: プロンプト設計 + 早期停止 (StoppingCriteria)

# %%
from transformers import StoppingCriteria, StoppingCriteriaList
import re

class StopOnPattern(StoppingCriteria):
    def __init__(self, tokenizer, input_len, stop_regex=r"[\n\.。！!？?]|$"):
        super().__init__()
        self.tok = tokenizer
        self.input_len = input_len
        self.re_pat = re.compile(stop_regex)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs):
        text = self.tok.decode(input_ids[0][self.input_len:], skip_special_tokens=True)
        return bool(self.re_pat.search(text))

topic = "生成AIに関して新しい研究を考えてください"
instr = (
    "次のお題に対して、新しいアイデアの要素となる英語キーワードのみをカンマ区切りで3〜5個出力。"
    "説明文や記号は不要。英小文字/数字/ハイフン/スペースのみ。\n"
    f"お題: {topic}\n"
    "keywords: "
)

inputs = tok(instr, return_tensors="pt").to(device)
stops = StoppingCriteriaList([StopOnPattern(tok, inputs['input_ids'].shape[1])])

out = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    stopping_criteria=stops,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)

print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

semantic


In [ ]:
# %% [markdown]
# # 3) 方式B: prefix_allowed_tokens_fn で正規表現の「前方一致」だけ許可（修正版）
# 生成文字列が常に ^word(, word)* のprefixになるよう許可トークンを絞る
# - word := [a-z0-9]+([ -][a-z0-9]+)*
# - 区切り := ", "
# ※ input_ids は 1D（形状: [seq_len]）で渡ってくる点に注意

# %%
import re

WORD = r"[a-z0-9]+(?:[ -][a-z0-9]+)*"
SEP  = r", "
FULL = rf"^{WORD}(?:{SEP}{WORD})*$"
PREFIX = rf"^(?:{WORD}(?:{SEP}{WORD})*)?$"   # 前方一致許容

prefix_re = re.compile(PREFIX)
full_re   = re.compile(FULL)

def build_prefix_allowed_regex_fn(tokenizer, prompt_len: int):
    vocab_size = tokenizer.vocab_size
    special_ids = set(getattr(tokenizer, "all_special_ids", []) or [])
    id2piece = {}

    # 速度対策で全トークンを一度decodeしてキャッシュ（特殊トークンは除外）
    for tid in range(vocab_size):
        if tid in special_ids:
            id2piece[tid] = ""
            continue
        try:
            s = tokenizer.decode([tid], skip_special_tokens=True)
        except Exception:
            s = ""
        id2piece[tid] = s

    def fn(batch_id: int, sent_ids):
        # sent_ids は 1D テンソル（形: [seq_len]）
        # すでに生成済みのテキスト
        tail_ids = sent_ids[prompt_len:]
        cur = tokenizer.decode(tail_ids, skip_special_tokens=True)

        # すでにprefix違反なら何も許可しない（=その枝を打ち切り）
        if not prefix_re.match(cur):
            return []

        allowed = []
        # 末尾が完全式（FULL）を満たしていれば EOS を許可
        if full_re.match(cur):
            allowed.append(tokenizer.eos_token_id)

        # 追加する各トークン片で prefix を保てるものだけ許可
        for tid, piece in id2piece.items():
            if not piece:
                continue
            # 改行やタブは許可しない
            if any(c in piece for c in "\n\r\t"):
                continue
            nxt = cur + piece
            if prefix_re.match(nxt):
                allowed.append(tid)

        return allowed

    return fn

topic = "生成AIに関して新しい研究を考えてください"
prompt = (
    "以下のお題について、新しいアイデアの要素となる '英語キーワード' のみを出力せよ。"
    "形式: lowercase keywords, comma separated（例: diffusion acceleration, robust finetuning）。\n"
    f"題目: {topic}\n"
    "keywords: "
)

inputs = tok(prompt, return_tensors="pt").to(device)
prefix_fn = build_prefix_allowed_regex_fn(tok, inputs["input_ids"].shape[1])

out = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    prefix_allowed_tokens_fn=prefix_fn,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)

print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
import os, sys, subprocess

# まず tokenizers の警告を消す（fork 安全）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# これが肝心：いま使っている Python の pip を使う
# /app/.venv/bin/pip など壊れたパスは無視して、常に sys.executable -m pip を使う
def pip_install(*pkgs):
    print("Using interpreter:", sys.executable)
    cmd = [sys.executable, "-m", "pip", "install", *pkgs]
    print(">", " ".join(cmd))
    subprocess.check_call(cmd)

# pip 自体をアップグレード（任意だが安定）
try:
    pip_install("--upgrade", "pip")
except Exception as e:
    print("pip upgrade failed (ignored):", repr(e))

# Outlines をインストール
pip_install("outlines")
print("✅ outlines installed")

Using interpreter: /app/.venv/bin/python
> /app/.venv/bin/python -m pip install --upgrade pip
Using interpreter: /app/.venv/bin/python
> /app/.venv/bin/python -m pip install outlines
  Using cached outlines-1.2.8-py3-none-any.whl.metadata (28 kB)
  Using cached cloudpickle-3.1.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached genson-1.3.0-py3-none-any.whl.metadata (28 kB)
  Using cached jsonpath_ng-1.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached ply-3.11-py2.py3-none-any.whl.metadata (844 bytes)
Using cached outlines-1.2.8-py3-none-any.whl (98 kB)
Using cached outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.3 MB)
Using cached cloudpickle-3.1.1-py3-none-any.whl (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Using cached genson-1.3.0-py3-none-any.whl (21 

In [ ]:
# %% [markdown]
# # 方式C-最終安定版（推奨）: CSV 3〜5語を強制（空配列フォールバック付き）
# - 出力例: diffusion acceleration, robust finetuning, multi-modal distillation
# - 文字集合: [a-z0-9 - ,]
# - 正規表現で prefix を維持しつつ、空になったら「許可文字のみのトークン」全体でフォールバック

# %%
import re

WORD = r"[a-z0-9]+(?:[ -][a-z0-9]+)*"
FULL = rf"^{WORD}(?:, {WORD}){{2,4}}$"                # 3〜5語
PREFIX = rf"^(?:{WORD}(?:, {WORD}){{0,4}}(?:, )?)?$"  # 前方一致を広めに

full_re   = re.compile(FULL)
prefix_re = re.compile(PREFIX)

def _build_token_whitelist(tokenizer):
    # 許可文字のみから成るトークンを抽出（改行/制御は除外）
    allowed_chars = set("abcdefghijklmnopqrstuvwxyz0123456789- ,")
    special_ids = set(getattr(tokenizer, "all_special_ids", []) or [])
    wl = []
    for tid in range(tokenizer.vocab_size):
        if tid in special_ids:
            continue
        tok = tokenizer.convert_ids_to_tokens(tid)
        if not isinstance(tok, str) or tok == "":
            continue
        # SentencePiece のマーカーなどを緩く正規化
        piece = tok.replace("▁", " ")
        if any(c in piece for c in "\n\r\t"):
            continue
        # 全て許可文字のみか？
        if all((c.lower() in allowed_chars) for c in piece.lower()):
            wl.append(tid)
    return wl

def build_prefix_allowed_csv_fn(tokenizer, prompt_len: int):
    whitelist = _build_token_whitelist(tokenizer)
    eos_id = tokenizer.eos_token_id

    def fn(batch_id: int, sent_ids):
        tail_ids = sent_ids[prompt_len:]
        cur = tokenizer.decode(tail_ids, skip_special_tokens=True)

        # 完成形なら EOS を許可
        allowed = [eos_id] if full_re.match(cur) else []

        # 既にprefix違反なら（通常は打ち切りだが）フォールバックで継続可能に
        if not prefix_re.match(cur):
            return allowed or whitelist or [eos_id]

        # 追加しても prefix を保てるトークンのみ許可
        cand = []
        for tid in whitelist:
            piece = tokenizer.convert_ids_to_tokens(tid).replace("▁", " ")
            nxt = cur + piece
            if prefix_re.match(nxt):
                cand.append(tid)

        # 空になったらフォールバック（常に非空を保証）
        return (allowed + cand) if cand or allowed else (whitelist or [eos_id])

    return fn

topic = "生成AIに関して新しい研究を考えてください"
prompt = (
    "Output only lowercase English keywords, comma separated, 3 to 5 items. "
    "Allowed chars: [a-z0-9 -]. No explanations.\n"
    f"Topic: {topic}\n"
    "keywords: "
)

inputs = tok(prompt, return_tensors="pt").to(device)
csv_prefix_fn = build_prefix_allowed_csv_fn(tok, inputs["input_ids"].shape[1])

out = model.generate(
    **inputs,
    max_new_tokens=96,
    do_sample=False, temperature=0.0, top_p=1.0,  # まずは決定的で挙動確認
    prefix_allowed_tokens_fn=csv_prefix_fn,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

machine-learning-generator-research-topic-generator-algorithm-neural-networks-comparison-analysis-generative-models-differentiation-training-methods-evolutionary-algorithms-genetic-algorithms-neural-symbolic-computation-combinatorial-optimization-problem-solving-strategy-randomized-search-algorithms-genetic-programming-genetic-algorithms-neural-symbolic-computation-combinatorial-optimization-problem-solving-strategy-randomized-search-algorithms-genetic-programming-genetic-algorithms-neural-symbol


In [ ]:
import re

# --- 3〜5語: lowercase + space/hyphen, comma separated ---
WORD = r"[a-z0-9]+(?:[ -][a-z0-9]+)*"
FULL = rf"^{WORD}(?:, {WORD}){{2,4}}$"                 # 完成形 (=3〜5語)
PREFIX = rf"^(?:{WORD}(?:, {WORD}){{0,4}}(?:, )?)?$"   # 途中形 (=前方一致)

full_re = re.compile(FULL)
prefix_re = re.compile(PREFIX)

def _build_token_whitelist(tokenizer):
    allowed = set("abcdefghijklmnopqrstuvwxyz0123456789-, ")
    special = set(getattr(tokenizer, "all_special_ids", []) or [])
    wl = []
    for tid in range(tokenizer.vocab_size):
        if tid in special:
            continue
        piece = tokenizer.convert_ids_to_tokens(tid).replace("▁", " ")
        if piece and all((c.lower() in allowed) for c in piece.lower()):
            wl.append(tid)
    return wl

def build_prefix_allowed_csv_fn(tokenizer, prompt_len):
    whitelist = _build_token_whitelist(tokenizer)
    eos = tokenizer.eos_token_id
    def fn(batch_id, sent_ids):
        cur = tokenizer.decode(sent_ids[prompt_len:], skip_special_tokens=True)
        allowed = [eos] if full_re.match(cur) else []
        if not prefix_re.match(cur):
            return allowed or whitelist or [eos]
        cand = []
        for tid in whitelist:
            nxt = cur + tokenizer.convert_ids_to_tokens(tid).replace("▁", " ")
            if prefix_re.match(nxt):
                cand.append(tid)
        return (allowed + cand) if cand or allowed else (whitelist or [eos])
    return fn


# ---- 実行例 ----
topic = "生成AIに関して新しい研究を考えてください"
prompt = (
    "Output only lowercase English keywords, comma separated, 3 to 5 items. "
    "Allowed chars: [a-z0-9 -]. No explanations.\n"
    f"Topic: {topic}\n"
    "keywords: "
)

inputs = tok(prompt, return_tensors="pt").to(device)
prefix_fn = build_prefix_allowed_csv_fn(tok, inputs["input_ids"].shape[1])

out = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=False, temperature=0.0, top_p=1.0,  # まずは決定論的
    prefix_allowed_tokens_fn=prefix_fn,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.eos_token_id,
)

result = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print(result)

machine-learning-generator-research-topic-generator-algorithm-neural-networks-comparison-analysis-generative-models-differentiation-training-methods-evolutionary-algorithms-genetic-algorithms-neural-symbolic-computation-combinatorial-optimization-problem-solving-strategy-randomized-search-algorithms-genetic-programming-genetic-algorithms


In [ ]:
# ===== JSON版（最終安定）: “]”で止める + バランス抽出 =====
import re, json

WORD = r"[a-z0-9]+(?:[ -][a-z0-9]+)*"
JSON_FULL   = re.compile(rf'^\[\s*"(?:{WORD})"(?:\s*,\s*"(?:{WORD})"){{2,4}}\s*\]$')
JSON_PREFIX = re.compile(
    r'^$|'
    r'^\[\s*(?:(?:"[a-z0-9 -]*")(?:\s*,\s*"[a-z0-9 -]*")*|\s*"[a-z0-9 -]*)?\s*\]?$'
)

def _wl_json(tokenizer):
    allowed = set('abcdefghijklmnopqrstuvwxyz0123456789- ,[]"')
    special = set(getattr(tokenizer, "all_special_ids", []) or [])
    wl = []
    for tid in range(tokenizer.vocab_size):
        if tid in special:
            continue
        piece = tokenizer.convert_ids_to_tokens(tid).replace("▁", " ")
        if piece and all(c.lower() in allowed for c in piece.lower()):
            wl.append(tid)
    return wl

def build_prefix_allowed_json_fn(tokenizer, prompt_len):
    wl = _wl_json(tokenizer)
    eos = tokenizer.eos_token_id
    def fn(batch_id, sent_ids):
        cur = tokenizer.decode(sent_ids[prompt_len:], skip_special_tokens=True)
        # 完成形になったら EOS のみを許可
        if JSON_FULL.match(cur):
            return [eos]
        # 逸脱したらフォールバック
        if not JSON_PREFIX.match(cur):
            return wl or [eos]
        cand = []
        for tid in wl:
            nxt = cur + tokenizer.convert_ids_to_tokens(tid).replace("▁", " ")
            if JSON_PREFIX.match(nxt):
                cand.append(tid)
        return cand or (wl or [eos])
    return fn

# “]” 自体でも終了させる（環境により ‘]’ が単独トークンであることが多い）
def right_bracket_eos_ids(tokenizer):
    ids = set()
    # 文字列 " ] " をトークナイズして最初のトークンを拾う（複数パターン試す）
    for seed in ["]", " ]", "] ", '"]', ' ] ']:
        ids_seed = tokenizer(seed, add_special_tokens=False).input_ids
        if ids_seed:
            ids.add(ids_seed[0])
    # さらに語彙内で “]” を含むトークンも加える
    for tid in range(tokenizer.vocab_size):
        try:
            piece = tokenizer.convert_ids_to_tokens(tid)
        except Exception:
            continue
        if isinstance(piece, str) and "]" in piece:
            ids.add(tid)
    return list(ids)

def extract_first_json_array_balanced(s: str):
    """最初のトップレベルJSON配列を厳密に切り出す（文字列内の [] は無視）"""
    start = s.find("[")
    if start == -1:
        return None
    i, n = start, len(s)
    level = 0
    in_str = False
    esc = False
    while i < n:
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "[":
                level += 1
            elif ch == "]":
                level -= 1
                if level == 0:
                    return s[start:i+1]
        i += 1
    return None  # 閉じられなかった

# ---- 実行例 ----
topic = "生成AIに関して新しい研究を考えてください.特にLLMが創造性を働かすことができる方法について考えてください．"
prompt = (
    "Return ONLY a JSON array of 10 lowercase English keywords.\n"
    f"Topic: {topic}\n"
    "JSON: "
)

inputs = tok(prompt, return_tensors="pt").to(device)
prefix_fn = build_prefix_allowed_json_fn(tok, inputs["input_ids"].shape[1])

# “]” での終了も受け付ける（複数EOSに対応）
eos_list = [tok.eos_token_id] + right_bracket_eos_ids(tok)

out = model.generate(
    **inputs,
    max_new_tokens=96,
    do_sample=False, temperature=0.0, top_p=1.0,
    prefix_allowed_tokens_fn=prefix_fn,
    eos_token_id=eos_list,                  # ← ここがポイント
    pad_token_id=tok.eos_token_id,
)

raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("RAW:", raw)

# まずは素直に読む → ダメならバランス抽出 → それでもダメなら失敗
try:
    arr = json.loads(raw)
except Exception:
    sub = extract_first_json_array_balanced(raw)
    if not sub:
        raise
    arr = json.loads(sub)

# 正規化 + 重複除去
norm = lambda s: re.sub(r'\s+',' ', re.sub(r'[^a-z0-9 -]','', s.lower().strip()))
arr = list(dict.fromkeys(norm(x) for x in arr if norm(x)))
print("CLEAN:", ", ".join(arr[:5]))

RAW: ["creativity-testing-methods-for-llms-in-text-generation-research"]
CLEAN: creativity-testing-methods-for-llms-in-text-generation-research
